##**Review Intelligence Pipeline - BERT Fine-Tuning**

This notebook fine-tunes a pretrained BERT model for 5-class sentiment classification on the Yelp Review Full dataset. Unlike BoW and LSTM approaches built from scratch,`BERT` (Bidirectional Encoder Representations from Transformers) comes pretrained on massive text corpora and understands deep contextual relationships between words — "not good" and "good" produce fundamentally different representations.

**Goal:** Fine-tune `bert-base-uncased` on a stratified subset of the Yelp dataset and evaluate whether transfer learning outperforms our previous baselines.

**Result:** 59.44% accuracy on 10k training samples — surpassing both the BoW (56%) and GloVe LSTM (56%) baselines despite using significantly less training data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers datasets accelerate -q
print("Setup Complete")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup Complete


##**Loading and Caching the Dataset**

Loading the Yelp Review Full dataset directly via the HuggingFace `datasets` library. The dataset is cached to Google Drive on first load — subsequent runs load from disk instantly, surviving runtime disconnects.

Link: [yelp_review_full](https://huggingface.co/datasets/Yelp/yelp_review_full)

In [ ]:
from datasets import load_dataset
import os

CACHE_PATH = '/content/drive/MyDrive/Colab Notebooks/yelp_dataset'

if os.path.exists(CACHE_PATH):
  from datasets import load_from_disk
  dataset = load_from_disk(CACHE_PATH)
  print("Loaded from drive")
else:
  dataset = load_dataset("Yelp/yelp_review_full")
  dataset.save_to_disk(CACHE_PATH)
  print("Downloaded and saved to drive")

print(dataset)

Loaded from drive
DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})


In [ ]:
dataset['train'][5]['text']

"Top notch doctor in a top notch practice. Can't say I am surprised when I was referred to him by another doctor who I think is wonderful and because he went to one of the best medical schools in the country. \\nIt is really easy to get an appointment. There is minimal wait to be seen and his bedside manner is great."

In [ ]:
dataset['train'][5]['label']

4

In [ ]:
dataset['train'][5]

{'label': 4,
 'text': "Top notch doctor in a top notch practice. Can't say I am surprised when I was referred to him by another doctor who I think is wonderful and because he went to one of the best medical schools in the country. \\nIt is really easy to get an appointment. There is minimal wait to be seen and his bedside manner is great."}

##**Tokenizer**

The BERT tokenizer converts raw text into numerical indices from BERT's fixed vocabulary of ~30,000 tokens. Unlike simple word tokenizers, it uses **WordPiece tokenization** — breaking unknown words into subword units (e.g. "playing" → "play" + "##ing").

Every sequence is wrapped with two special tokens: `[CLS]` at the start (used for classification) and `[SEP]` at the end. The tokenizer returns a dictionary with three keys:

- **`input_ids`** — token indices from BERT's vocabulary
- **`attention_mask`** — 1 for real tokens, 0 for padding
- **`token_type_ids`** — distinguishes sentence A from sentence B in two-sentence tasks; all zeros here since we only have one sentence per sample

We test on a single sample first to verify the output format before building the full dataset pipeline.

In [ ]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
text = dataset['train'][5]['text']
encoded = tokenizer(text, padding=True, truncation=True, return_tensors='pt') #padding=True pads shorted sequnces to match the lonest in the batch
#truncation=True — cuts sequences longer than 512 tokens
#return_tensors='pt' — returns PyTorch tensors instead of plain lists
print(encoded)
#input_ids are words being converted to tokens from BERT's vocabulary
#attetnion_maks tells BERT which tokens to pay attention to...1 means important and 0 simply means padding
#token_type_ids used for tasks with two sentences (e.g. Q&A). All zeros here since we only have one sentence.

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

{'input_ids': tensor([[  101,  2327, 18624,  3460,  1999,  1037,  2327, 18624,  3218,  1012,
          2064,  1005,  1056,  2360,  1045,  2572,  4527,  2043,  1045,  2001,
          3615,  2000,  2032,  2011,  2178,  3460,  2040,  1045,  2228,  2003,
          6919,  1998,  2138,  2002,  2253,  2000,  2028,  1997,  1996,  2190,
          2966,  2816,  1999,  1996,  2406,  1012,  1032,  9152,  2102,  2003,
          2428,  3733,  2000,  2131,  2019,  6098,  1012,  2045,  2003, 10124,
          3524,  2000,  2022,  2464,  1998,  2010, 19475,  5450,  2003,  2307,
          1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [ ]:
print(encoded['attention_mask'].shape) #Shape [1, 72] — 1 sample, 72 tokens (including [CLS] and [SEP])

torch.Size([1, 72])


In [ ]:
tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])
print(tokens)

['[CLS]', 'top', 'notch', 'doctor', 'in', 'a', 'top', 'notch', 'practice', '.', 'can', "'", 't', 'say', 'i', 'am', 'surprised', 'when', 'i', 'was', 'referred', 'to', 'him', 'by', 'another', 'doctor', 'who', 'i', 'think', 'is', 'wonderful', 'and', 'because', 'he', 'went', 'to', 'one', 'of', 'the', 'best', 'medical', 'schools', 'in', 'the', 'country', '.', '\\', 'ni', '##t', 'is', 'really', 'easy', 'to', 'get', 'an', 'appointment', '.', 'there', 'is', 'minimal', 'wait', 'to', 'be', 'seen', 'and', 'his', 'bedside', 'manner', 'is', 'great', '.', '[SEP]']


##**Yelp Dataset Class**
To feed data into a PyTorch model, we need to wrap it in a custom `Dataset` class. This class is responsible for tokenizing each review on-the-fly during training rather than tokenizing the entire dataset upfront (which would be memory-intensive).

Three methods are required:
- **`__init__`** — stores the raw dataset and tokenizer
- **`__len__`** — returns the total number of samples
- **`__getitem__`** — tokenizes and returns a single sample by index

`padding='max_length'` is used instead of `padding=True` to ensure every sample is padded to exactly 512 tokens — this guarantees all tensors in a batch are the same size, which is required for the DataLoader to stack them. `torch.squeeze()` removes the extra batch dimension the tokenizer adds when `return_tensors='pt'` is specified.

In [ ]:
import torch
from torch.utils.data import Dataset

class YelpDataset(Dataset):
  """
  This will convert each of our review in the dataset to tokens from
  the existing BERT library.
  """
  def __init__(self, data, tokenizer):
    self.data = data
    self.tokenizer = tokenizer

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    tokenized = self.tokenizer(self.data[idx]['text'], max_length=512, padding='max_length', truncation=True, return_tensors='pt')
    return {"input_ids":torch.squeeze(tokenized['input_ids']),
            "attention_mask":torch.squeeze(tokenized['attention_mask']),
            "label":torch.tensor(self.data[idx]['label'])}

##**Stratified Data Subset**

Fine-tuning BERT on all 650k samples would take days on a free Colab T4 GPU. We instead use a small stratified subset — 2000 samples per class for training (10k total) and 500 per class for testing (2500 total).

Stratified sampling is important here: a random slice of the dataset could be class-imbalanced, skewing both training and evaluation. By explicitly filtering per class and selecting equal counts, we ensure the model sees a balanced distribution across all 5 sentiment labels.

In [ ]:
from datasets import concatenate_datasets
def get_stratified_subset(split: Dataset, per_class: int) -> Dataset:
  """
  Breaks down entire dataset into a filtered subset(list of dataset), containing 2000
  samples per sample. At the end, elements of this subset(many datasets) are
  concatenated to create one train/testing dataset.
  """
  subset = []
  for label in range(5):
    filtered = split.filter(lambda x: x['label']==label)
    subset.append(filtered.select(range(per_class)))
  return concatenate_datasets(subset)

In [ ]:
train_subset = get_stratified_subset(split = dataset['train'], per_class=2000)
test_subset = get_stratified_subset(split = dataset['test'], per_class=500)

In [ ]:
train_dataset = YelpDataset(train_subset, tokenizer)
test_dataset = YelpDataset(test_subset, tokenizer)

print(len(train_dataset))
print(len(test_dataset))

10000
2500


##**DataLoaders**

PyTorch `DataLoader` wraps the dataset and handles batching, shuffling, and parallel data loading during training. We use `batch_size=16` — a conservative choice given BERT's size (110M parameters) and the T4 GPU's 15GB VRAM limit. Larger batch sizes risk CUDA out-of-memory errors.

Training loader uses `shuffle=True` to randomize sample order each epoch, preventing the model from learning label order patterns. Test loader uses `shuffle=False` to keep evaluation deterministic.

In [ ]:
from torch.utils.data import DataLoader
train_loader = DataLoader(dataset=train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=16, shuffle=False)
print(len(train_loader))

625


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


##**Loading BERT**

We use `bert-base-uncased` — a 12-layer, 768-hidden, 110M parameter transformer pretrained on BooksCorpus and English Wikipedia. `BertForSequenceClassification` adds a linear classification head (`Linear(768 → 5)`) on top of the pretrained encoder.

The pretrained encoder weights remain and are fine-tuned during training.

In [ ]:
from transformers import BertForSequenceClassification
BERTV0 = BertForSequenceClassification.from_pretrained(pretrained_model_name_or_path='bert-base-uncased', num_labels=5).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
print(BERTV0)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

##**Optimizer and Scheduler**

**AdamW** (`lr=2e-5`) is the standard optimizer for fine-tuning BERT. It's a variant of Adam with decoupled weight decay, which prevents large weight values and improves generalization. `2e-5` is a deliberately small learning rate — too large and the pretrained weights get destroyed in the first few updates.

**Linear warmup scheduler** gradually increases the learning rate from 0 to `2e-5` over the first 312 steps (warmup), then linearly decays it back to 0 by the end of training. This protects the pretrained representations from large gradient updates early on and helps the model converge smoothly.

Total training steps = 5 epochs × 625 batches = 3125. Warmup steps = ~10% = 312.

In [ ]:
import transformers
optimizer = torch.optim.AdamW(params=BERTV0.parameters(), lr=2e-5)
scheduler = transformers.get_linear_schedule_with_warmup(optimizer, num_training_steps=5*len(train_loader),num_warmup_steps=312)

##**Training Loop**

For each batch: move tensors to GPU, run forward pass, compute loss, backpropagate, clip gradients to `max_norm=1.0` to prevent exploding gradients, then step the optimizer and scheduler. Loss is printed every 100 batches to monitor progress.

In [ ]:
BERTV0.train()
epochs = 5
for epoch in range(epochs):
  train_loss = 0
  for batch_idx, batch in enumerate(train_loader): #here, batch will be a dictionary
    input_ids = batch['input_ids'].to(device) #enumerate helps loop through any iterable, list, tuple anythin
    attention_mask = batch['attention_mask'].to(device)
    label = batch['label'].to(device)
    optimizer.zero_grad()
    output = BERTV0(input_ids, attention_mask=attention_mask, labels=label)
    train_loss += output.loss.item()
    output.loss.backward()
    torch.nn.utils.clip_grad_norm_(BERTV0.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()
    if batch_idx%100==0:
      print(f"Epoch: {epoch} | Batch: {batch_idx} | Loss: {train_loss/(batch_idx+1)}")



Epoch: 0 | Batch: 0 | Loss: 1.6797785758972168
Epoch: 0 | Batch: 100 | Loss: 1.6310615067434784
Epoch: 0 | Batch: 200 | Loss: 1.5225408735559947
Epoch: 0 | Batch: 300 | Loss: 1.3973288171711158
Epoch: 0 | Batch: 400 | Loss: 1.3099620822361877
Epoch: 0 | Batch: 500 | Loss: 1.2519525579825608
Epoch: 0 | Batch: 600 | Loss: 1.2019564532202214
Epoch: 1 | Batch: 0 | Loss: 0.8253757953643799
Epoch: 1 | Batch: 100 | Loss: 0.8179620085376325
Epoch: 1 | Batch: 200 | Loss: 0.8253685590343096
Epoch: 1 | Batch: 300 | Loss: 0.8190313574681647
Epoch: 1 | Batch: 400 | Loss: 0.8180124873383682
Epoch: 1 | Batch: 500 | Loss: 0.8146372175145292
Epoch: 1 | Batch: 600 | Loss: 0.8175453970118886
Epoch: 2 | Batch: 0 | Loss: 0.8653139472007751
Epoch: 2 | Batch: 100 | Loss: 0.6138984699650566
Epoch: 2 | Batch: 200 | Loss: 0.6142410800528171
Epoch: 2 | Batch: 300 | Loss: 0.6201870920650191
Epoch: 2 | Batch: 400 | Loss: 0.6187134990073796
Epoch: 2 | Batch: 500 | Loss: 0.6142126389011413
Epoch: 2 | Batch: 600 | Lo

##**Evaluation Loop**

Model is set to `eval()` mode and `torch.inference_mode()` disables gradient computation, reducing memory usage and speeding up inference. For each batch we collect true labels and predicted labels (via `argmax` on logits), then concatenate all batches at the end for scoring.

In [ ]:
BERTV0.eval()
true_labels = []
pred_labels = []
test_loss = 0
with torch.inference_mode():
  for batch_idx, batch in enumerate(test_loader):
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    label = batch['label'].to(device)
    true_labels.append(label.tolist())
    y_pred = BERTV0(input_ids, attention_mask=attention_mask, labels=label)
    test_loss += y_pred.loss.item()
    predictions = torch.argmax(y_pred.logits, dim=1)
    pred_labels.append(predictions.cpu().tolist())
    if batch_idx % 100 == 0:
      print(f"Batch: {batch_idx} | Test Loss: {test_loss/(batch_idx+1)}")
print(f"Final Test Loss: {test_loss/len(test_loader)}")
import numpy as np
true_labels = np.concatenate(true_labels)
pred_labels = np.concatenate(pred_labels)

Batch: 0 | Test Loss: 0.9838483929634094
Batch: 100 | Test Loss: 1.312691073016365
Final Test Loss: 1.2673893269080265


In [ ]:
from sklearn.metrics import accuracy_score, classification_report
acc = accuracy_score(true_labels, pred_labels)
print("Accuracy: ",acc)

print(classification_report(true_labels, pred_labels))

Accuracy:  0.5944
              precision    recall  f1-score   support

           0       0.72      0.70      0.71       500
           1       0.51      0.56      0.53       500
           2       0.53      0.53      0.53       500
           3       0.53      0.48      0.50       500
           4       0.69      0.70      0.70       500

    accuracy                           0.59      2500
   macro avg       0.60      0.59      0.59      2500
weighted avg       0.60      0.59      0.59      2500



##**Results and Analysis**

**Test Accuracy: 59.44%** on 2500 balanced test samples (500 per class), trained on only 10k samples.

This surpasses both previous baselines — BoW Logistic Regression (56%) and GloVe LSTM (56%) — despite using significantly less training data, demonstrating the power of transfer learning.

Key observations from the classification report:
- **Classes 0 and 4** (1-star and 5-star) achieve the highest F1 scores (0.71 and 0.70) — extreme sentiments are linguistically distinct and easier to separate
- **Classes 1, 2, 3** (middle ratings) are frequently confused with each other — the boundary between a 2-star and 3-star review is often subtle even for humans

**Next steps:** Scale training data to 20k-50k samples. The architecture is correct — more data is the primary bottleneck.